# Лабораторная работа №10: RAG BM25

[Исходное задание](https://colab.research.google.com/drive/1lnhGZhzz1Y6iAkI2bVA-mvJiDLb3by3W)

Выполнение через совместимый API RouterAI. API-ключ берётся из окружения или локального `.env`. Сохранённые результаты получены реальными запросами.



## Задание

##Задание на семинар
Известны следующие подходы, используемые в RAG-системах: мультиплицирование запроса и суммаризация чанков.

**Мультиплицирование запроса** - подход, при котором запрос от пользователя несколько раз перефразируется с помощью LLM и осуществляется поиск чанков по всем вариантам запроса. Результаты поиска объединяют в один.

**Суммаризация чанков**  - подход, при котором найденные чанки «упрощаются» с помощью LLM (например, создается выжимка), а затем подаются на вход в LLM (добавляются к запросу пользователя).

1. Реализуйте мультиплицирование запроса, когда запрос от пользователя перефразируется LLM 3-5 раз и для каждого варианта запроса осуществляется поиск чанков. Дубликаты чанков удаляются программным кодом. Найденные чанки подаются LLM (в цепочку). Перефразирование LLM реализуйте с помощью цепочки. Используйте ретривер BM25. Принцип деление на чанки - по абзацам.
1. Реализуйте суммаризацию чанков. Упрощение (выжимка) реализуйте с помощью цепочки.

## Подключение

In [ ]:
from pathlib import Path
import sys

# При открытии из подпапки находим общий модуль конфигурации.
for candidate in [Path.cwd(), Path.cwd().parent]:
    if (candidate / "lab_runtime.py").exists():
        sys.path.insert(0, str(candidate))
        break

from lab_runtime import make_llm, json_result, ASSETS
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, Markdown, Image, Audio

llm = make_llm()
print("Модель:", llm.model_name)


## 1. База знаний и чанки по абзацам

In [ ]:
import re
import hashlib
from rank_bm25 import BM25Okapi

knowledge = """RAG объединяет поиск и генерацию. Найденные сведения добавляются в контекст вопроса, чтобы модель отвечала по базе знаний.

BM25 — алгоритм поиска документов по словам запроса. Он учитывает частоту терминов, редкость слова в коллекции и длину документа.

Мультиплицирование запроса создаёт несколько перефразировок исходного вопроса. Поиск по разным формулировкам повышает полноту найденной информации.

Суммаризация чанков сжимает найденную информацию. Выжимка уменьшает объём контекста, сохраняя факты, важные для ответа.

Разбиение по абзацам сохраняет завершённые мысли. Каждый непустой абзац становится отдельным чанком.

Дубликаты найденных чанков удаляются программно по хешу текста. Это предотвращает повторение одинакового контекста.

Ответ RAG должен опираться на найденный контекст. Если сведений недостаточно, ассистент сообщает об этом и не выдумывает факты."""
chunks = [p.strip() for p in knowledge.split("\n\n") if p.strip()]
tokenize = lambda text: re.findall(r"[а-яёa-z0-9]+", text.lower())
bm25 = BM25Okapi([tokenize(chunk) for chunk in chunks])

def retrieve(query, k=3):
    scores = bm25.get_scores(tokenize(query))
    indices = sorted(range(len(chunks)), key=lambda i: scores[i], reverse=True)[:k]
    return [chunks[i] for i in indices if scores[i] > 0]

print("Чанков:", len(chunks))


## 2. Четыре перефразировки, поиск и удаление дубликатов

In [ ]:
query_chain = ChatPromptTemplate.from_messages([
    ("system", 'Верни JSON-объект вида {{"queries":["строка 1","строка 2","строка 3","строка 4"]}} с четырьмя разными перефразировками вопроса. Каждый элемент queries — строка, не объект. Используй термины RAG, чанки, BM25, поиск, контекст, суммаризация, если подходят.'),
    ("human", "{question}")
]) | llm.bind(response_format={"type":"json_object"}) | StrOutputParser()
question = "Как повысить полноту поиска в RAG и уменьшить размер контекста?"
variants = json_result(query_chain.invoke({"question": question}))["queries"]
assert len(variants) == 4
assert all(isinstance(query, str) for query in variants)
raw_found = [chunk for query in [question] + variants for chunk in retrieve(query)]
unique = {}
for chunk in raw_found:
    digest = hashlib.sha256(chunk.encode()).hexdigest()
    unique[digest] = chunk
found = list(unique.values())
print("Перефразировки:", *variants, sep="\n")
print("До удаления дубликатов:", len(raw_found), "После:", len(found))
print("Контекст:", *found, sep="\n\n")

answer_chain = ChatPromptTemplate.from_messages([
    ("system", "Отвечай только по контексту. До 120 слов, укажи использованные факты."),
    ("human", "Контекст: {context}\nВопрос: {question}")
]) | llm | StrOutputParser()
answer = answer_chain.invoke({"context": "\n\n".join(found), "question": question})
print("Ответ RAG:", answer)
assert found


## 3. Суммаризация найденных чанков отдельной цепочкой

In [ ]:
summary_chain = PromptTemplate.from_template("Сделай выжимку контекста до 80 слов для ответа на вопрос {question}. Не потеряй различие поиска и суммаризации. Контекст: {context}") | llm | StrOutputParser()
context = "\n\n".join(found)
summary = summary_chain.invoke({"question": question, "context": context})
final = answer_chain.invoke({"question": question, "context": summary})
print("Символов в контексте:", len(context), "в выжимке:", len(summary))
print("Выжимка:", summary, "\nИтог:", final)


## Вывод

Чанки разделены по абзацам. LLM-цепочка создала четыре формулировки вопроса. BM25 нашёл чанки, SHA-256 удалил дубликаты. Отдельная цепочка сжала контекст перед итоговым ответом.